In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.datasets import load_digits
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler

## Preprocessing


In [27]:
df = pd.read_csv("heart.csv")

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


In [24]:
df.describe()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


In [25]:
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [31]:
num_col = df.select_dtypes(include=np.number).columns
for col in num_col:
  upper = df[col].mean() + 3 * df[col].std()
  lower = df[col].mean() - 3 * df[col].std()
  df = df[(df[col] < upper) & (df[col] > lower)]
df

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1


In [32]:
df["Sex"] = df["Sex"].map({
    "M": 0,
    "F": 1
})
df["ST_Slope"] = df["ST_Slope"].map({
    "Up": 1,
    "Flat": 2,
    "Down": 3
})
df["ExerciseAngina"] = df["ExerciseAngina"].map({
    "N": 0,
    "Y": 1
})
df["RestingECG"] = df["RestingECG"].map({
    "Normal": 1,
    "LVH": 2,
    "ST": 3
})
df

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,0,ATA,140,289,0,1,172,0,0.0,1,0
1,49,1,NAP,160,180,0,1,156,0,1.0,2,1
2,37,0,ATA,130,283,0,3,98,0,0.0,1,0
3,48,1,ASY,138,214,0,1,108,1,1.5,2,1
4,54,0,NAP,150,195,0,1,122,0,0.0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,0,TA,110,264,0,1,132,0,1.2,2,1
914,68,0,ASY,144,193,1,1,141,0,3.4,2,1
915,57,0,ASY,130,131,0,1,115,1,1.2,2,1
916,57,1,ATA,130,236,0,2,174,0,0.0,2,1


In [33]:
df = pd.get_dummies(df, drop_first=True)
df

,Age,Sex,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA
0,40,0,140,289,0,1,172,0,0.0,1,0,True,False,False
1,49,1,160,180,0,1,156,0,1.0,2,1,False,True,False
2,37,0,130,283,0,3,98,0,0.0,1,0,True,False,False
3,48,1,138,214,0,1,108,1,1.5,2,1,False,False,False
4,54,0,150,195,0,1,122,0,0.0,1,0,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,0,110,264,0,1,132,0,1.2,2,1,False,False,True
914,68,0,144,193,1,1,141,0,3.4,2,1,False,False,False
915,57,0,130,131,0,1,115,1,1.2,2,1,False,False,False
916,57,1,130,236,0,2,174,0,0.0,2,1,True,False,False


In [50]:
x = df.drop("HeartDisease",axis='columns')
y = df.HeartDisease

In [53]:
x = StandardScaler().fit_transform(x)

## Choosing the best model

In [58]:
parameter_model = {
    'log_reg': {
        'model': LogisticRegression(solver='liblinear'),
        'params': {
            'C': [0.1, 1] + [i for i in range(5, 21, 5)]
        }
    },
    'random_forest': {
        'model': RandomForestClassifier(),
        'params': {
            'n_estimators': [5] + [i for i in range(10,51,10)]
        }
    },
    'svm': {
        'model': SVC(gamma='auto'),
        'params': {
            'C': [0.1, 1] + [i for i in range(5, 21,5)]
        }
    }
}

In [64]:
def evaluate(x, y):
  scores = []
  for model_name, mp in parameter_model.items():
    clf = GridSearchCV(mp['model'], mp['params'], cv=5, return_train_score=False)
    clf.fit(x,y)
    scores.append({
      'model': model_name,
      'best_score': clf.best_score_,
      'best_params': clf.best_params_
  })
  return scores

In [66]:
df1 = pd.DataFrame(evaluate(x, y), columns=['model', 'best_score', 'best_params'])
df1

,model,best_score,best_params
0,log_reg,0.810844,{'C': 0.1}
1,random_forest,0.823048,{'n_estimators': 20}
2,svm,0.825307,{'C': 0.1}


# PCA

In [61]:
from sklearn.decomposition import PCA

In [63]:
pca = PCA(0.95)
x_pca = pca.fit_transform(x)
x_pca.shape

(899, 12)

In [67]:
df2 = pd.DataFrame(evaluate(x_pca, y), columns=['model', 'best_score', 'best_params'])
df2

,model,best_score,best_params
0,log_reg,0.805270,{'C': 0.1}
1,random_forest,0.827523,{'n_estimators': 30}
2,svm,0.821943,{'C': 1}
